# Ачча! — озвучка фраз туриста без акцента (Kokoro)

Этот блокнот озвучивает **только фразы туриста** открытой моделью **Kokoro-82M** (лицензия Apache 2.0) — чистый американский или британский английский. Реплики индийцев остаются из первого блокнота (Indic Parler-TTS).

Видеокарта и токен Hugging Face **не нужны**. Запускайте ячейки по порядку кнопкой ▶. Всё вместе — минут 5.

## 1. Установка (1–2 минуты)

In [ ]:
!pip -q install "kokoro>=0.9.4" soundfile
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null
print('Готово, переходите к шагу 2')

## 2. Настройки

- `ACCENT`: `'a'` — американский английский, `'b'` — британский
- `VOICE`: для американского — `af_heart` (женский, самый естественный), `am_michael` (мужской); для британского — `bf_emma` (женский), `bm_george` (мужской)
- `SPEED`: `1.0` — обычный темп. Приложение и так проигрывает записи немного медленнее, поэтому лучше оставить 1.0
- `FIX` — исправления произношения: слева как во фразе, справа как прочитать модели
- `ONLY_IDS` — переозвучить только эти фразы, например `['air-01']`; `[]` — все фразы туриста

In [ ]:
PHRASES_URL = 'https://achha-india.github.io/app/phrases.json'
ACCENT = 'a'
VOICE = 'af_heart'
SPEED = 1.0
FIX = {
    'e-visa': 'e-veeza',
}
ONLY_IDS = []
OUT_DIR = '/content/drive/MyDrive/Achha/audio-tourist'
print('Настройки сохранены')

## 3. Модель и фразы

In [ ]:
import json, urllib.request, os, subprocess, re
import numpy as np, soundfile as sf
from kokoro import KPipeline
pipe = KPipeline(lang_code=ACCENT)
SR = 24000
data = json.load(urllib.request.urlopen(PHRASES_URL))
lines = [l for st in data['stages'] for s in st['situations'] for l in s['lines']]
todo = [l for l in lines if l['role'] == 'tourist']
if ONLY_IDS: todo = [l for l in lines if l['id'] in ONLY_IDS]
print('Фраз туриста к озвучке:', len(todo))

def synth(text):
    for k, v in FIX.items():
        text = re.sub(re.escape(k), v, text, flags=re.I)
    parts = [a for _, _, a in pipe(text, voice=VOICE, speed=SPEED)]
    a = np.concatenate([p.cpu().numpy() if hasattr(p, 'cpu') else np.asarray(p) for p in parts]).astype(np.float32)
    nz = np.where(np.abs(a) > 0.01)[0]
    if len(nz): a = a[max(0, nz[0] - int(0.05 * SR)): nz[-1] + int(0.15 * SR)]
    return a

def to_mp3(audio, path):
    wav = path.replace('.mp3', '.wav')
    sf.write(wav, audio, SR)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-ac', '1', '-b:a', '64k', path], check=True)
    os.remove(wav)
print('Готово')

## 4. Пробное прослушивание

Если голос не нравится — поменяйте `VOICE` (и `ACCENT`) на шаге 2 и запустите шаги 2, 3 и 4 ещё раз.

In [ ]:
from IPython.display import Audio, display
if 'todo' not in globals():
    raise RuntimeError('Сначала запустите шаг 2 и шаг 3 — шаг 3 должен закончиться словом «Готово»')
for l in todo[:3]:
    print(l['id'], '·', l['en'])
    display(Audio(synth(l['en']), rate=SR))

## 5. Озвучить всё и сохранить в Google Drive

Colab попросит доступ к Google Drive — разрешите. Файлы появятся в папке **Achha/audio-tourist**, архив **achha-audio-tourist.zip** скачается сам. Его пришлите в чат.

In [ ]:
from google.colab import drive, files
import shutil
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
failed = []
for i, l in enumerate(todo, 1):
    path = f"{OUT_DIR}/{l['id']}.mp3"
    if os.path.exists(path) and not ONLY_IDS: continue
    try:
        to_mp3(synth(l['en']), path); print(f"[{i}/{len(todo)}] {l['id']} ✓")
    except Exception as e:
        failed.append(l['id']); print(f"[{i}/{len(todo)}] {l['id']} ошибка: {e}")
zip_path = shutil.make_archive('/content/drive/MyDrive/Achha/achha-audio-tourist', 'zip', OUT_DIR)
print('\nГотово | ошибки:', failed or 'нет')
files.download(zip_path)